# Evaluate a RAG application with LangSmith (Luna edition)

A notebook version of the [LangSmith RAG evaluation tutorial](https://docs.langchain.com/langsmith/evaluate-rag-tutorial), with OpenAI's **GPT-6 Luna** replacing `gpt-5.5` for both answering and judging.

You'll:
1. Build a small RAG app over three Lilian Weng blog posts
2. Create a LangSmith dataset of question / reference-answer pairs
3. Define four LLM-as-judge evaluators: **correctness**, **relevance**, **groundedness**, **retrieval relevance**
4. Run an experiment and inspect the results in LangSmith

### API keys you need
| Key | Where to get it | Used for |
|---|---|---|
| `LANGSMITH_API_KEY` | smith.langchain.com → Settings → API Keys | Tracing, dataset, experiments |
| `OPENAI_API_KEY` | platform.openai.com → API keys | GPT Luna (answers + the judge evaluators) and embeddings |

Create a `.env` file next to this notebook:
```
LANGSMITH_API_KEY=lsv2_...
OPENAI_API_KEY=sk-...
LANGSMITH_TRACING=true
# Optional: LANGSMITH_PROJECT=rag-eval-tutorial
# Optional, EU region: LANGSMITH_ENDPOINT=https://eu.api.smith.langchain.com
```

## 1. Setup

In [1]:
#%pip install -qU langsmith "langchain[openai]" langchain-core langchain-community langchain-text-splitters bs4 requests python-dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current working directory
os.environ.setdefault("LANGSMITH_TRACING", "true")

REQUIRED = ("LANGSMITH_API_KEY", "OPENAI_API_KEY")
missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise EnvironmentError(f"Missing in .env: {', '.join(missing)}")
print("Keys loaded ✔")

Keys loaded ✔


Models used throughout. **GPT-6 Luna** (`gpt-6-luna`) is OpenAI's fast, low-cost GPT-6 tier; it replaces the tutorial's `gpt-5.5` for both answering and judging. Use `gpt-5.6-luna` if you want the previous generation.

`GEN_TEMPERATURE` / `JUDGE_TEMPERATURE` are `None` (model default) because reasoning models often accept only the default temperature. The tutorial used 1 and 0; set those values if Luna accepts them.

In [3]:
from langchain_openai import ChatOpenAI

LUNA_MODEL = "gpt-6-luna"
EMBED_MODEL = "text-embedding-3-small"
GEN_TEMPERATURE = None
JUDGE_TEMPERATURE = None

def luna(temperature=None) -> ChatOpenAI:
    kwargs = {} if temperature is None else {"temperature": temperature}
    return ChatOpenAI(model=LUNA_MODEL, **kwargs)

# Connectivity check
print(luna().invoke("Reply with the single word: ready").content)

ready


## 2. Build the RAG application
### Index the documents
Load three blog posts, split them into ~250-token chunks and store them in an in-memory vector store.

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [doc for sub in docs for doc in sub]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(model=EMBED_MODEL),
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
print(f"Indexed {len(doc_splits)} chunks from {len(docs_list)} pages")

C:\Users\Roberto\AppData\Local\Temp\ipykernel_53768\8649722.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Indexed 187 chunks from 3 pages


### Generate answers
`@traceable` sends each call to LangSmith. The bot returns both the answer and the retrieved documents, because the groundedness and retrieval-relevance evaluators need the documents.

In [5]:
from langsmith import traceable

llm = luna(GEN_TEMPERATURE)

@traceable()
def rag_bot(question: str) -> dict:
    docs = retriever.invoke(question)
    docs_string = "\n\n".join(doc.page_content for doc in docs)
    instructions = (
        "You are a helpful assistant who is good at analyzing source information "
        "and answering questions. Use the following source documents to answer "
        "the user's question. If you don't know the answer, just say that you don't know. "
        "Use three sentences maximum and keep the answer concise.\n\n"
        f"Documents:\n{docs_string}"
    )
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])
    return {"answer": ai_msg.content, "documents": docs}

In [6]:
# Quick smoke test
out = rag_bot("What is task decomposition in LLM agents?")
print(out["answer"])

Task decomposition is when an LLM agent breaks a complex task into smaller, manageable subgoals or steps. It can do this through prompting (such as “think step by step”), task-specific instructions, or human input, making complex tasks easier to plan and complete.


## 3. Create a dataset
Three questions with reference answers. The cell is safe to re-run: it only creates the dataset if it doesn't exist yet.

In [7]:
from langsmith import Client

client = Client()

examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    },
]

dataset_name = "Lilian Weng Blogs Q&A"
if client.has_dataset(dataset_name=dataset_name):
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Using existing dataset: {dataset.id}")
else:
    dataset = client.create_dataset(dataset_name=dataset_name)
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print(f"Created dataset: {dataset.id}")

Using existing dataset: 8cb57415-b146-4242-a61b-2908b473b049


## 4. Define evaluators
Each evaluator is an LLM-as-judge that returns structured output: an explanation (reasoning first, which tends to improve grading) and a boolean score.

| Evaluator | Compares | Needs reference? |
|---|---|---|
| Correctness | answer ↔ reference answer | Yes |
| Relevance | answer ↔ question | No |
| Groundedness | answer ↔ retrieved docs | No |
| Retrieval relevance | retrieved docs ↔ question | No |

All four judges run on GPT-6 Luna. Each evaluator returns its score **plus the judge's explanation as a comment**, so you can read why a judge passed or failed an answer directly next to the score in LangSmith.

In [8]:
from typing_extensions import Annotated, TypedDict

def make_grader(schema):
    return luna(JUDGE_TEMPERATURE).with_structured_output(
        schema, method="json_schema", strict=True
    )

### Correctness: is the answer right compared to the reference?

In [9]:
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

correctness_instructions = """You are a teacher grading a quiz. You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER.

Grading criteria:
(1) Grade the student answer based ONLY on its factual accuracy relative to the ground truth answer.
(2) Make sure the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to it.

Correctness: True means the student answer meets all the criteria. False means it does not.
Explain your reasoning step by step, then give the score. Avoid simply stating the correct answer at the outset."""

correctness_llm = make_grader(CorrectnessGrade)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """An evaluator for RAG answer accuracy."""
    content = (
        f"QUESTION: {inputs['question']}\n"
        f"GROUND TRUTH ANSWER: {reference_outputs['answer']}\n"
        f"STUDENT ANSWER: {outputs['answer']}"
    )
    grade = correctness_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": content},
    ])
    return {"key": "correctness", "score": grade["correct"], "comment": grade["explanation"]}

### Relevance: does the answer address the question?

In [10]:
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a STUDENT ANSWER.

Grading criteria:
(1) The student answer is concise and relevant to the question.
(2) The student answer helps to answer the question.

Relevance: True means the student answer meets all the criteria. False means it does not.
Explain your reasoning step by step, then give the score."""

relevance_llm = make_grader(RelevanceGrade)

def relevance(inputs: dict, outputs: dict) -> dict:
    """A simple evaluator for RAG answer helpfulness."""
    content = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": content},
    ])
    return {"key": "relevance", "score": grade["relevant"], "comment": grade["explanation"]}

### Groundedness: is the answer supported by the retrieved docs (no hallucinations)?

In [11]:
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

grounded_instructions = """You are a teacher grading a quiz. You will be given FACTS and a STUDENT ANSWER.

Grading criteria:
(1) The student answer is grounded in the FACTS.
(2) The student answer does not contain "hallucinated" information outside the scope of the FACTS.

Grounded: True means the student answer meets all the criteria. False means it does not.
Explain your reasoning step by step, then give the score."""

grounded_llm = make_grader(GroundedGrade)

def groundedness(inputs: dict, outputs: dict) -> dict:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    content = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": content},
    ])
    return {"key": "groundedness", "score": grade["grounded"], "comment": grade["explanation"]}

### Retrieval relevance: are the retrieved docs relevant to the question?

In [12]:
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

retrieval_relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a set of FACTS provided by the student.

Grading criteria:
(1) Identify FACTS that are completely unrelated to the QUESTION.
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant.
(3) It is OK if the facts have SOME information unrelated to the question, as long as (2) is met.

Relevance: True means the FACTS contain ANY keywords or semantic meaning related to the QUESTION. False means they are completely unrelated.
Explain your reasoning step by step, then give the score."""

retrieval_relevance_llm = make_grader(RetrievalRelevanceGrade)

def retrieval_relevance(inputs: dict, outputs: dict) -> dict:
    """An evaluator for document relevance."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    content = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": content},
    ])
    return {"key": "retrieval_relevance", "score": grade["relevant"], "comment": grade["explanation"]}

## 5. Run the evaluation
The target function maps a dataset example's `inputs` to the app's outputs. LangSmith runs it on every example and applies all four evaluators.

In [13]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance-luna",
    metadata={"model": LUNA_MODEL},
)

View the evaluation results for experiment: 'rag-doc-relevance-luna-023675a0' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/8cb57415-b146-4242-a61b-2908b473b049/compare?selectedSessions=3034225b-8e5f-45bb-afd6-ad8968765ac0




0it [00:00, ?it/s]

In [14]:
# Results as a DataFrame (needs pandas); the full experiment is also linked above in LangSmith
df = experiment_results.to_pandas()
feedback_cols = [c for c in df.columns if c.startswith("feedback.")]
display(df[["inputs.question", "outputs.answer", *feedback_cols]])
df[feedback_cols].mean()

,inputs.question,outputs.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance
0,How does the ReAct agent use self-reflection?,ReAct does not use an explicit self-reflection...,True,True,True,True
1,What are the types of biases that can arise wi...,Few-shot prompting can exhibit **majority-labe...,True,True,True,True
2,What are five types of adversarial attacks?,"Five types are token manipulation, gradient-ba...",True,True,True,True


feedback.correctness            1.0
feedback.groundedness           1.0
feedback.relevance              1.0
feedback.retrieval_relevance    1.0
dtype: float64

## Next steps
- Open the experiment link printed above to see per-example scores and the judge's explanations.
- Change `chunk_size`, `k` or the prompt, re-run section 5, and compare experiments side by side in LangSmith.
- Add more examples to the dataset to make the scores meaningful (3 is just a demo).

## 6. Add more examples to the dataset
Three examples are just a demo: one wrong judgement moves a score by 33 points. Below are 12 more question / reference-answer pairs, 4 per blog post, written from the posts themselves. The cell only adds questions that aren't in the dataset yet, so it's safe to re-run.

In [15]:
more_examples = [
    # --- LLM Powered Autonomous Agents ---
    {"inputs": {"question": "What are the three main components that complement the LLM in an LLM-powered autonomous agent?"},
     "outputs": {"answer": "Planning (subgoal decomposition and reflection/refinement), memory (short-term and long-term), and tool use (calling external APIs)."}},
    {"inputs": {"question": "In LLM agents, what do short-term memory and long-term memory correspond to?"},
     "outputs": {"answer": "Short-term memory is in-context learning, limited by the context window; long-term memory is an external vector store the agent can query with fast retrieval."}},
    {"inputs": {"question": "In what three ways can task decomposition be done for an LLM agent?"},
     "outputs": {"answer": "(1) By the LLM with simple prompting such as 'Steps for XYZ', (2) with task-specific instructions such as 'Write a story outline', or (3) with human inputs."}},
    {"inputs": {"question": "What are the four stages of HuggingGPT?"},
     "outputs": {"answer": "Task planning, model selection, task execution, and response generation."}},
    # --- Prompt Engineering ---
    {"inputs": {"question": "What is prompt engineering?"},
     "outputs": {"answer": "Prompt engineering (in-context prompting) is about how to communicate with an LLM to steer its behavior toward desired outcomes without updating the model weights; it is an empirical science that needs heavy experimentation."}},
    {"inputs": {"question": "What is the difference between zero-shot and few-shot prompting?"},
     "outputs": {"answer": "Zero-shot gives the model only the task text, while few-shot first shows a set of high-quality input-output demonstrations; few-shot often performs better but uses more tokens and may hit the context length limit."}},
    {"inputs": {"question": "What is self-consistency sampling?"},
     "outputs": {"answer": "Sampling multiple outputs with temperature > 0 and then selecting the best one, usually by majority vote (or, for verifiable tasks like code, by running tests)."}},
    {"inputs": {"question": "What are the two main types of chain-of-thought prompting?"},
     "outputs": {"answer": "Few-shot CoT, where demonstrations include step-by-step reasoning, and zero-shot CoT, which uses a phrase like 'Let's think step by step' without demonstrations."}},
    # --- Adversarial Attacks on LLMs ---
    {"inputs": {"question": "What is the difference between white-box and black-box adversarial attacks?"},
     "outputs": {"answer": "White-box attacks assume full access to the model weights and architecture, so gradients can be used (only possible for open models); black-box attacks only have API-like access, sending inputs and getting outputs."}},
    {"inputs": {"question": "Which threat model do gradient-based attacks require?"},
     "outputs": {"answer": "The white-box setting: they need full access to the model parameters and architecture, so they only work on models like open-source LLMs."}},
    {"inputs": {"question": "According to Wei et al. (2023), what are the two failure modes of LLM safety training?"},
     "outputs": {"answer": "Competing objectives (helpfulness conflicts with safety, e.g. prefix injection or refusal suppression) and mismatched generalization (safety training doesn't cover inputs the model still understands, e.g. Base64 or other encodings)."}},
    {"inputs": {"question": "How does red-teaming with a red LM work?"},
     "outputs": {"answer": "A red-team LM generates test prompts, the target model responds, and a classifier detects harmful outputs; the red LM can be used zero-shot, few-shot, supervised fine-tuned, or trained with RL."}},
]

existing = {ex.inputs["question"].strip() for ex in client.list_examples(dataset_id=dataset.id)}
to_add = [e for e in more_examples if e["inputs"]["question"].strip() not in existing]
if to_add:
    client.create_examples(dataset_id=dataset.id, examples=to_add)
print(f"Added {len(to_add)} examples -> dataset now has {len(existing) + len(to_add)}")

Added 12 examples -> dataset now has 15


## 7. Compare `chunk_size`, `k` and the prompt
Each configuration below rebuilds the retriever, runs the full dataset (now 15 examples) and becomes its own experiment in LangSmith, with the settings stored as metadata. The baseline is re-run too, so every experiment uses the same 15 examples and is directly comparable.

| Config | chunk_size | k | Prompt | What it tests |
|---|---|---|---|---|
| baseline | 250 | 6 | default | the tutorial's settings |
| small-k | 250 | 2 | default | less context: does retrieval miss things? |
| big-chunks | 500 | 4 | default | fewer, larger chunks (about the same amount of text) |
| strict-prompt | 250 | 6 | strict | answer only from the docs, use their wording for lists |

Each config makes about 75 Luna calls (15 answers × 5) and takes a few minutes.

In [16]:
import pandas as pd

DEFAULT_INSTRUCTIONS = (
    "You are a helpful assistant who is good at analyzing source information "
    "and answering questions. Use the following source documents to answer "
    "the user's question. If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
)
STRICT_INSTRUCTIONS = (
    "You are a helpful assistant who answers questions strictly from the source documents below. "
    "Only state facts that appear in the documents; if they don't contain the answer, say that you don't know. "
    "When the question asks for a list, types or steps, give the complete list using the names the documents use. "
    "Use three sentences maximum and keep the answer concise."
)
PROMPTS = {"default": DEFAULT_INSTRUCTIONS, "strict": STRICT_INSTRUCTIONS}

_stores = {}  # chunk_size -> vector store (so each chunk size is only embedded once)

def build_retriever(chunk_size: int, k: int):
    if chunk_size not in _stores:
        splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=chunk_size, chunk_overlap=0)
        _stores[chunk_size] = InMemoryVectorStore.from_documents(
            documents=splitter.split_documents(docs_list),
            embedding=OpenAIEmbeddings(model=EMBED_MODEL),
        )
    return _stores[chunk_size].as_retriever(search_kwargs={"k": k})

def make_target(retriever, instructions: str):
    @traceable(name="rag_bot")
    def bot(question: str) -> dict:
        docs = retriever.invoke(question)
        docs_string = "\n\n".join(doc.page_content for doc in docs)
        ai_msg = llm.invoke([
            {"role": "system", "content": f"{instructions}\n\nDocuments:\n{docs_string}"},
            {"role": "user", "content": question},
        ])
        return {"answer": ai_msg.content, "documents": docs}
    return lambda inputs: bot(inputs["question"])

CONFIGS = [
    {"name": "baseline",      "chunk_size": 250, "k": 6, "prompt": "default"},
    {"name": "small-k",       "chunk_size": 250, "k": 2, "prompt": "default"},
    {"name": "big-chunks",    "chunk_size": 500, "k": 4, "prompt": "default"},
    {"name": "strict-prompt", "chunk_size": 250, "k": 6, "prompt": "strict"},
]

all_results, rows = {}, []
for cfg in CONFIGS:
    print(f"\n=== {cfg['name']} ===")
    results = client.evaluate(
        make_target(build_retriever(cfg["chunk_size"], cfg["k"]), PROMPTS[cfg["prompt"]]),
        data=dataset_name,
        evaluators=[correctness, groundedness, relevance, retrieval_relevance],
        experiment_prefix=f"rag-{cfg['name']}",
        metadata={"model": LUNA_MODEL, **cfg},
        max_concurrency=4,
    )
    all_results[cfg["name"]] = results
    df_cfg = results.to_pandas()
    scores = df_cfg[[c for c in df_cfg.columns if c.startswith("feedback.")]].astype(float).mean()
    rows.append({"config": cfg["name"], "experiment": results.experiment_name, **scores.round(2).to_dict()})

comparison = pd.DataFrame(rows).set_index("config")
comparison.columns = [c.replace("feedback.", "") for c in comparison.columns]
comparison


=== baseline ===
View the evaluation results for experiment: 'rag-baseline-5326adb7' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/8cb57415-b146-4242-a61b-2908b473b049/compare?selectedSessions=86953931-3d5a-4853-8a29-99bf95f0b18b




0it [00:00, ?it/s]


=== small-k ===
View the evaluation results for experiment: 'rag-small-k-88ddbcdc' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/8cb57415-b146-4242-a61b-2908b473b049/compare?selectedSessions=f0bdb16a-6000-4395-ae23-7cd646f37a4b




0it [00:00, ?it/s]


=== big-chunks ===
View the evaluation results for experiment: 'rag-big-chunks-bfe0a753' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/8cb57415-b146-4242-a61b-2908b473b049/compare?selectedSessions=1f77ced2-285e-41bf-a8f0-93b2b5f2b2b4




0it [00:00, ?it/s]


=== strict-prompt ===
View the evaluation results for experiment: 'rag-strict-prompt-723018f8' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/8cb57415-b146-4242-a61b-2908b473b049/compare?selectedSessions=308ae3cf-42c3-4417-b1fd-d32f13b99ea2




0it [00:00, ?it/s]

,experiment,correctness,groundedness,relevance,retrieval_relevance
config,,,,,
baseline,rag-baseline-5326adb7,0.93,0.87,1.00,1.0
small-k,rag-small-k-88ddbcdc,0.93,0.80,0.93,1.0
big-chunks,rag-big-chunks-bfe0a753,0.80,0.80,0.87,1.0
strict-prompt,rag-strict-prompt-723018f8,0.87,0.93,1.00,1.0


**Compare them side by side in LangSmith:** open *Datasets & Experiments → Lilian Weng Blogs Q&A*, tick the four `rag-…` experiments and click **Compare**. Each row is one question, so you can see exactly which question a config gets right or wrong, and click any score to read the judge's comment.

## 8. Read the judges' explanations
Every failed score, with the judge's reason, for each configuration. These are the cases worth checking by hand: sometimes the answer is wrong, sometimes the judge is.

In [17]:
for name, results in all_results.items():
    print(f"\n=== {name} ===")
    for r in results:
        question = r["example"].inputs["question"]
        for ev in r["evaluation_results"]["results"]:
            if ev.score is not None and not ev.score:
                print(f"✗ {ev.key:<20} | {question}\n    judge: {ev.comment}\n")


=== baseline ===
✗ groundedness         | What are the four stages of HuggingGPT?
    judge: The facts explicitly identify task planning as stage 1 and response generation as stage 4. They do not identify stages 2 and 3 as model selection and task execution. Those may be part of the system, but they are not supported by the provided facts, so the answer includes unsupported information.

✗ correctness          | What is the difference between white-box and black-box adversarial attacks?
    judge: The black-box description matches the ground truth: the attacker sends inputs through an API and observes outputs without access to the model’s internals. The white-box description correctly mentions access to weights and architecture and the use of gradients, but it adds access to the training pipeline as part of the definition. That access is not required by the ground truth’s definition of a white-box attack, so the answer overstates the assumption.

✗ groundedness         | What are the 